# 02. Entailment dan Inference

Notebook ini membedakan **syntax**, **semantics**, dan **model**, lalu memakai ketiganya untuk memahami **logical entailment**. Kita akan mengenumerasi delapan kemungkinan dunia Wumpus, menentukan dunia yang sesuai dengan pengetahuan agent, dan melihat kapan sebuah kesimpulan benar-benar dijamin oleh knowledge base (KB).

Setelah menyelesaikan notebook ini, Anda diharapkan mampu:

1. membedakan bentuk kalimat, makna kalimat, dan model yang memenuhi kalimat;
2. menjelaskan $KB \vDash \alpha$ melalui relasi himpunan model;
3. memeriksa entailment dengan enumerasi model; dan
4. membedakan algoritma inferensi yang **sound** dan **complete**.

## Setup

Jalankan sel berikut sekali di awal. Sel ini mencari environment AIMA yang disertakan di dalam repo dan memuat pustaka yang diperlukan. Jika notebook dibuka melalui Google Colab, repo akan di-clone secara otomatis.

Environment siap ketika baris terakhir mencetak `Check       : tt_entails(P & Q, Q) = True`.

In [1]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

Environment : /home/ata/school/Modul-Praktikum-KK-RKA-25/logics/praktikum/environment
Python      : 3.12.3
Check       : tt_entails(P & Q, Q) = True


---
# 2.1 Syntax, Semantics, dan Model

## Penjelasan

| Istilah | Pertanyaan yang dijawab | Contoh |
|---|---|---|
| **Syntax** | Apakah susunan simbol ini merupakan kalimat yang valid? | `x + y = 4` *well-formed*, sedangkan `x3y+ =` tidak. |
| **Semantics** | Apa makna kalimat itu, dan kapan ia benar? | `x + y = 4` benar ketika nilai `x` dan `y` berjumlah 4. |
| **Model** | Pada keadaan dunia yang mana kalimat itu benar? | Penetapan `x = 2, y = 2` merupakan salah satu model dari `x + y = 4`. |

Syntax hanya memeriksa **bentuk**, bukan kebenaran. Sebuah sentence dapat tersusun secara valid tetapi bernilai salah pada suatu model. Semantics memberikan aturan untuk menentukan nilai kebenaran sentence pada setiap model. Jika sentence $\alpha$ benar pada model $m$, kita tulis

$$m \vDash \alpha,$$

dibaca "$m$ memenuhi $\alpha$". Himpunan seluruh model yang memenuhi $\alpha$ ditulis sebagai

$$M(\alpha) = \{m \mid m \vDash \alpha\}.$$

> **Catatan istilah:** model di sini adalah satu kemungkinan keadaan dunia atau interpretasi simbol, bukan model hasil pelatihan dalam *machine learning*.

## Logical entailment

Sentence $\alpha$ **meng-entail** sentence $\beta$, ditulis $\alpha \vDash \beta$, jika $\beta$ benar di **setiap** model tempat $\alpha$ benar. Definisi formalnya adalah

$$\boxed{\alpha \vDash \beta \quad\text{jika dan hanya jika}\quad M(\alpha) \subseteq M(\beta)}.$$

Ketika premisnya berupa knowledge base, notasinya menjadi $KB \vDash \alpha$. Kata pentingnya adalah **setiap**: satu contoh yang cocok belum cukup untuk membuktikan entailment, tetapi satu *counterexample*, model tempat KB benar dan kesimpulan salah, cukup untuk membantahnya.

## Contoh penerapan

Contoh aritmatika berikut memisahkan syntax dari semantics. Semua ekspresi `x + y = 4` memiliki syntax yang sama; nilai kebenarannya berubah menurut model atau penetapan nilai variabel.

In [2]:
# Each dictionary represents one model for x and y.
arith_models = [
    {"x": 2, "y": 2},
    {"x": 1, "y": 1},
    {"x": 0, "y": 4},
    {"x": 3, "y": 1},
]

rows = []
for model in arith_models:
    rows.append({
        **model,
        "x + y": model["x"] + model["y"],
        "x + y = 4": model["x"] + model["y"] == 4,
    })

arith_table = pd.DataFrame(rows)
arith_table

,x,y,x + y,x + y = 4
0,2,2,4,True
1,1,1,2,False
2,0,4,4,True
3,3,1,4,True


In [3]:
models_of_sentence = [
    model for model in arith_models
    if model["x"] + model["y"] == 4
]
print("M(x + y = 4) in the example domain:")
for model in models_of_sentence:
    print(" ", model)

M(x + y = 4) in the example domain:
  {'x': 2, 'y': 2}
  {'x': 0, 'y': 4}
  {'x': 3, 'y': 1}


Tabel menunjukkan bahwa syntax sentence tetap sama, tetapi nilai kebenarannya bergantung pada model. Dalam domain contoh, tiga penetapan nilai memenuhi `x + y = 4`, sedangkan model `x = 1, y = 1` tidak memenuhinya.

---
# 2.2 Membangun Ruang Model Wumpus

## Penjelasan

Agent telah mengunjungi `[1,1]` tanpa merasakan *breeze*, lalu berpindah ke `[2,1]` dan merasakan *breeze*. Percept tersebut digabungkan dengan aturan Wumpus World di dalam KB. Agent ingin mengetahui isi tiga kotak yang belum diketahui: `[1,2]`, `[2,2]`, dan `[3,1]`.

![Tiga kotak yang isinya belum diketahui](img/slide-18-tiga-kotak-belum-diketahui.png)

Kita gunakan simbol $P_{x,y}$ yang berarti "ada pit di $[x,y]$". Informasi yang relevan dapat disederhanakan menjadi:

- tidak ada *breeze* di `[1,1]`, sehingga tidak ada pit di `[1,2]`: $\neg P_{1,2}$;
- ada *breeze* di `[2,1]`, sehingga sedikitnya salah satu dari `[2,2]` atau `[3,1]` berisi pit: $P_{2,2} \lor P_{3,1}$.

Kotak `[1,1]` dan `[2,1]` tidak perlu divariasikan: keduanya sudah dikunjungi dan agent masih hidup. Jadi, untuk tiga simbol yang belum diketahui, bentuk ringkas KB adalah

$$KB = \neg P_{1,2} \land (P_{2,2} \lor P_{3,1}).$$

Setiap simbol dapat bernilai `False` atau `True`, sehingga terdapat $2^3=8$ possible model.

Gambar berikut memperlihatkan delapan kombinasi tersebut. Tugas model checking adalah menyaring mana yang benar-benar konsisten dengan KB.

![Delapan possible model untuk tiga kotak](img/slide-19-delapan-model.png)

## Contoh penerapan

Kode berikut membangkitkan kedelapan model dengan `itertools.product`, lalu menguji model mana yang konsisten dengan KB.

In [4]:
SYMBOLS = ("P12", "P22", "P31")

# product generates every False/True combination for the three symbols.
possible_models = [
    dict(zip(SYMBOLS, values))
    for values in itertools.product([False, True], repeat=len(SYMBOLS))
]

print(f"Number of possible models: {len(possible_models)}")
possible_models[:3]

Number of possible models: 8


[{'P12': False, 'P22': False, 'P31': False},
 {'P12': False, 'P22': False, 'P31': True},
 {'P12': False, 'P22': True, 'P31': False}]

In [5]:
def kb_wumpus(model):
    """Return True when a model satisfies ¬P12 ∧ (P22 ∨ P31)."""
    no_breeze_at_11 = not model["P12"]
    breeze_at_21 = model["P22"] or model["P31"]
    return no_breeze_at_11 and breeze_at_21

model_table = pd.DataFrame([
    {**model, "KB": kb_wumpus(model)}
    for model in possible_models
])
model_table.index = [f"m{i}" for i in range(1, len(model_table) + 1)]
model_table

,P12,P22,P31,KB
m1,False,False,False,False
m2,False,False,True,True
m3,False,True,False,True
m4,False,True,True,True
m5,True,False,False,False
m6,True,False,True,False
m7,True,True,False,False
m8,True,True,True,False


In [6]:
kb_models = [model for model in possible_models if kb_wumpus(model)]

assert len(possible_models) == 8
assert len(kb_models) == 3

print(f"Models satisfying KB: {len(kb_models)} of {len(possible_models)}")
for model in kb_models:
    print(" ", model)

Models satisfying KB: 3 of 8
  {'P12': False, 'P22': False, 'P31': True}
  {'P12': False, 'P22': True, 'P31': False}
  {'P12': False, 'P22': True, 'P31': True}


Tepat tiga model memenuhi KB: pit berada di `[3,1]` saja, di `[2,2]` saja, atau di kedua kotak tersebut. Semua model dengan `P12=True` ditolak karena bertentangan dengan tidak adanya *breeze* di `[1,1]`. Model tanpa pit di `[2,2]` maupun `[3,1]` juga ditolak karena tidak dapat menjelaskan *breeze* di `[2,1]`.

---
# 2.3 Menguji Dua Kesimpulan

## Penjelasan

Sekarang agent mempertimbangkan dua sentence:

- $\alpha_1 = \neg P_{1,2}$: "tidak ada pit di `[1,2]`";
- $\alpha_2 = \neg P_{2,2}$: "tidak ada pit di `[2,2]`".

![Figure 7.5: Relasi himpunan model KB, alpha 1, dan alpha 2](img/fig-7-5-model-alpha1-alpha2.png)

Pada Figure 7.5, seluruh $M(KB)$ berada di dalam $M(\alpha_1)$, sehingga $KB \vDash \alpha_1$. Sebaliknya, sebagian $M(KB)$ berada di luar $M(\alpha_2)$, sehingga $KB \nvDash \alpha_2$.

## Contoh penerapan

Tabel berikut memeriksa ketiga sentence pada seluruh delapan model; baris dengan `KB=True` menjadi fokus pengujian entailment.

In [7]:
def alpha_1(model):
    return not model["P12"]

def alpha_2(model):
    return not model["P22"]

comparison_table = pd.DataFrame([
    {
        **model,
        "KB": kb_wumpus(model),
        "α1 = ¬P12": alpha_1(model),
        "α2 = ¬P22": alpha_2(model),
        "¬α2 = P22": not alpha_2(model),
    }
    for model in possible_models
])
comparison_table.index = [f"m{i}" for i in range(1, len(comparison_table) + 1)]
comparison_table

,P12,P22,P31,KB,α1 = ¬P12,α2 = ¬P22,¬α2 = P22
m1,False,False,False,False,True,True,False
m2,False,False,True,True,True,True,False
m3,False,True,False,True,True,False,True
m4,False,True,True,True,True,False,True
m5,True,False,False,False,False,True,False
m6,True,False,True,False,False,True,False
m7,True,True,False,False,False,False,True
m8,True,True,True,False,False,False,True


In [8]:
def entails(kb, alpha, models):
    """Return True when alpha holds in every model satisfying kb."""
    return all(not kb(model) or alpha(model) for model in models)

checks = {
    "KB ⊨ α1": entails(kb_wumpus, alpha_1, possible_models),
    "KB ⊨ α2": entails(kb_wumpus, alpha_2, possible_models),
    "KB ⊨ ¬α2": entails(kb_wumpus, lambda m: not alpha_2(m), possible_models),
}

for statement, result in checks.items():
    print(f"{statement:<10}: {result}")

assert list(checks.values()) == [True, False, False]

KB ⊨ α1   : True
KB ⊨ α2   : False
KB ⊨ ¬α2  : False


In [9]:
# One counterexample is enough to establish KB ⊭ α2.
counterexamples_alpha_2 = [
    model for model in possible_models
    if kb_wumpus(model) and not alpha_2(model)
]
counterexamples_alpha_2

[{'P12': False, 'P22': True, 'P31': False},
 {'P12': False, 'P22': True, 'P31': True}]

Hasil `KB ⊭ α2` **bukan** berarti $\alpha_2$ pasti salah. Artinya, KB belum cukup untuk menjamin $\alpha_2$. Bahkan $KB \nvDash \neg\alpha_2$ juga: di antara model-model KB, ada dunia dengan pit di `[2,2]` dan ada dunia tanpa pit di sana.

Dengan kata lain, status `[2,2]` masih **unknown**. Logika membedakan tiga keadaan epistemik yang penting:

| Kondisi | Status kesimpulan |
|---|---|
| $KB \vDash \alpha$ | $\alpha$ pasti benar menurut KB |
| $KB \vDash \neg\alpha$ | $\alpha$ pasti salah menurut KB |
| $KB \nvDash \alpha$ dan $KB \nvDash \neg\alpha$ | KB belum menentukan nilai $\alpha$ |

---
# 2.4 Logical Inference

## Penjelasan

**Entailment** adalah relasi matematis antara sentence: apakah kesimpulan benar di semua model premis. **Inference** adalah proses atau algoritma yang mencoba menurunkan kesimpulan tersebut. Notasi yang umum adalah

$$KB \vDash \alpha \quad\text{(entailment)}$$

$$KB \vdash_i \alpha \quad\text{(algoritma inferensi }i\text{ menurunkan }\alpha\text{)}.$$

Metode pada bagian sebelumnya disebut **model checking**: enumerasi semua possible model, pilih model yang memenuhi KB, lalu periksa apakah $\alpha$ benar pada semuanya.

![Figure 7.6: Hubungan dunia, representasi, dan logical reasoning](img/fig-7-6-logical-reasoning.png)

Sentence di dalam agent merepresentasikan keadaan dunia. Logical reasoning mengubah sentence lama menjadi sentence baru; jaminan formalnya memastikan bahwa representasi baru memang mengikuti dari pengetahuan sebelumnya.

## Sound dan complete

Dua sifat berikut menghubungkan hasil algoritma ($\vdash_i$) dengan kebenaran semantis ($\vDash$):

| Sifat | Jaminan formal | Makna praktis | Jika tidak dimiliki |
|---|---|---|---|
| **Sound** (*truth-preserving*) | Jika $KB \vdash_i \alpha$, maka $KB \vDash \alpha$ | Semua kesimpulan yang dikeluarkan memang dijamin oleh KB. | Algoritma dapat menghasilkan kesimpulan salah (*false positive*), misalnya menyebut kotak berpit sebagai aman. |
| **Complete** | Jika $KB \vDash \alpha$, maka $KB \vdash_i \alpha$ | Semua kesimpulan yang secara logis mengikuti pada akhirnya dapat ditemukan. | Algoritma dapat melewatkan kesimpulan benar (*false negative*), misalnya gagal mengenali kotak yang sebenarnya dapat dibuktikan aman. |

Algoritma yang sound tetapi tidak complete bersikap konservatif: jawabannya dapat kurang lengkap, tetapi yang berhasil disimpulkan tetap benar. Algoritma yang complete tetapi tidak sound dapat menemukan seluruh konsekuensi benar sekaligus mengeluarkan konsekuensi yang tidak valid. Untuk agent keselamatan-kritis, pelanggaran soundness biasanya lebih berbahaya karena agent dapat bertindak berdasarkan klaim yang keliru.

Model checking yang benar-benar memeriksa seluruh ruang model proposisional bersifat **sound dan complete**. Kekurangannya adalah biaya: dengan $n$ simbol proposisi terdapat $2^n$ model. Masalah pertumbuhan ini akan dibahas lebih lanjut pada Notebook 04.

## Contoh penerapan

Fungsi `entails` dari bagian sebelumnya sekarang diuji dengan dua simbol sederhana agar sifat pemeriksaan modelnya terlihat tanpa mengulang contoh Wumpus.

In [10]:
# Use two symbols in an example separate from Wumpus World.
pq_models = [
    dict(zip(("P", "Q"), values))
    for values in itertools.product([False, True], repeat=2)
]

kb_p_and_q = lambda model: model["P"] and model["Q"]
alpha_p = lambda model: model["P"]
alpha_not_p = lambda model: not model["P"]

print("P ∧ Q ⊨ P  :", entails(kb_p_and_q, alpha_p, pq_models))
print("P ∧ Q ⊨ ¬P :", entails(kb_p_and_q, alpha_not_p, pq_models))

P ∧ Q ⊨ P  : True
P ∧ Q ⊨ ¬P : False


Hasil pertama `True` karena setiap model yang memenuhi $P \land Q$ juga memenuhi $P$. Hasil kedua `False` karena tidak ada model $P \land Q$ yang sekaligus memenuhi $\neg P$.

In [11]:
# Each additional symbol doubles the number of models.
growth = pd.DataFrame({
    "number_of_symbols (n)": range(1, 11),
    "number_of_models (2^n)": [2**n for n in range(1, 11)],
})
growth

,number_of_symbols (n),number_of_models (2^n)
0,1,2
1,2,4
2,3,8
3,4,16
4,5,32
5,6,64
6,7,128
7,8,256
8,9,512
9,10,1024


Tabel memperlihatkan pertumbuhan eksponensial: penambahan satu simbol selalu menggandakan jumlah model. Sepuluh simbol saja sudah menghasilkan 1.024 model yang harus diperiksa.

---
# Ringkasan

- **Syntax** menentukan sentence mana yang tersusun valid; **semantics** menentukan artinya dan kapan sentence benar.
- **Model** adalah satu interpretasi atau kemungkinan dunia. $M(\alpha)$ adalah himpunan model yang memenuhi $\alpha$.
- $KB \vDash \alpha$ tepat ketika $M(KB) \subseteq M(\alpha)$, tidak ada model tempat KB benar tetapi $\alpha$ salah.
- `KB ⊭ α` hanya menyatakan bahwa $\alpha$ belum dijamin; itu tidak otomatis berarti $\alpha$ salah.
- **Soundness** mencegah kesimpulan tidak valid, sedangkan **completeness** menjamin tidak ada konsekuensi valid yang terlewat.
- Exhaustive model checking sound dan complete, tetapi ruang pencariannya tumbuh secara eksponensial.

---
# Latihan Soal

## Soal 1: Memahami ketidakpastian

Jelaskan perbedaan antara "$\alpha$ salah di semua model KB" dan "$\alpha$ tidak benar di semua model KB". Kaitkan jawaban Anda dengan $\alpha_2 = \neg P_{2,2}$ pada Bagian 2.3.

<details>
<summary><strong>Hint Soal 1</strong></summary>

"$\alpha$ salah di semua model KB" berarti $KB \vDash \neg\alpha$: KB menjamin negasinya. "$\alpha$ tidak benar di semua model KB" hanya berarti $KB \nvDash \alpha$: sedikitnya ada satu counterexample, tetapi mungkin juga ada model KB lain tempat $\alpha$ benar.

Untuk $\alpha_2$, satu model KB membuat $\alpha_2$ benar dan dua model KB membuatnya salah. Karena itu $KB \nvDash \alpha_2$ sekaligus $KB \nvDash \neg\alpha_2$. Isi `[2,2]` masih belum diketahui.

</details>

## Soal 2: Membuktikan entailment dengan tabel model

Buktikan atau bantah $P \vDash P \lor Q$. Tulis kode untuk mengenumerasi empat model `P` dan `Q`, tampilkan nilai premis dan kesimpulan pada setiap model, lalu cari apakah ada *counterexample*.

<details>
<summary><strong>Hint Soal 2</strong></summary>

```python
models = [
    dict(zip(("P", "Q"), values))
    for values in itertools.product([False, True], repeat=2)
]

table = pd.DataFrame([
    {**m, "P ∨ Q": m["P"] or m["Q"]}
    for m in models
])
counterexamples = [m for m in models if m["P"] and not (m["P"] or m["Q"])]
display(table)
print("Counterexample:", counterexamples)
```

Daftar *counterexample* kosong. Setiap model yang membuat $P$ benar pasti membuat $P \lor Q$ benar, apa pun nilai $Q$. Jadi $P \vDash P \lor Q$.

</details>

## Soal 3: Kesimpulan yang belum ditentukan

Berikan satu contoh sederhana untuk $KB$ dan $\alpha$ yang memenuhi $KB \nvDash \alpha$ sekaligus $KB \nvDash \neg\alpha$. Verifikasi dengan fungsi `entails`, lalu jelaskan artinya bagi agent yang harus mengambil keputusan.

<details>
<summary><strong>Hint Soal 3</strong></summary>

Ambil $KB=P$ dan $\alpha=Q$.

```python
kb = lambda m: m["P"]
alpha = lambda m: m["Q"]
print(entails(kb, alpha, pq_models))                   # False
print(entails(kb, lambda m: not alpha(m), pq_models)) # False
```

Ketika `P=True`, `Q` masih dapat `True` atau `False`; KB tidak memberikan informasi tentang $Q$. Agent tidak boleh memperlakukan salah satu nilai sebagai kepastian. Ia perlu mencari percept tambahan, memilih aksi yang aman pada kedua kemungkinan, atau memakai kriteria keputusan lain di luar entailment.

</details>

## Soal 4: Soundness dan completeness

Bandingkan dua algoritma inferensi berikut untuk agent Wumpus:

1. Algoritma A sound tetapi tidak complete.
2. Algoritma B complete tetapi tidak sound.

Apa yang dapat terjadi pada masing-masing agent? Mana yang umumnya lebih berbahaya jika hasil inferensi langsung digunakan untuk bergerak?

<details>
<summary><strong>Hint Soal 4</strong></summary>

Algoritma A tidak pernah menyatakan sesuatu yang tidak dijamin KB, tetapi dapat gagal menemukan kesimpulan yang valid. Agent mungkin terlalu ragu, melewatkan rute aman, atau membutuhkan eksplorasi tambahan.

Algoritma B dapat menemukan semua konsekuensi valid, tetapi juga dapat mengeluarkan kesimpulan yang tidak mengikuti dari KB. Agent bisa menganggap kotak berpit sebagai aman. Jika hasil digunakan langsung untuk bergerak, pelanggaran soundness pada B umumnya lebih berbahaya daripada ketidaklengkapan A.

</details>